In [ ]:
# Get dependent repos from https://libraries.io/api

In [ ]:
import os

API_KEY: str = os.getenv("LIBRARIES_API_KEY")

! uv pip install requests pandas

Using Python 3.12.11 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Audited 2 packages in 2ms


In [ ]:
# Get supported package managers: https://libraries.io/platforms
# Looks like Pypi and Conda would be relevant

import requests

response = requests.get(
    "https://libraries.io/api/platforms", timeout=10, params={"api_key": API_KEY}
)
response.raise_for_status()

print([project["name"] for project in response.json()])

['NPM', 'Maven', 'Pypi', 'Go', 'NuGet', 'Packagist', 'Cargo', 'Rubygems', 'CocoaPods', 'Pub', 'Bower', 'CPAN', 'CRAN', 'Clojars', 'Conda', 'Hex', 'Hackage', 'Meteor', 'Homebrew', 'Puppet', 'Carthage', 'SwiftPM', 'Elm', 'Julia', 'Dub', 'Racket', 'Nimble', 'Haxelib', 'PureScript', 'Alcatraz', 'Inqlude']


In [ ]:
# Get dependent repositories
# NOTE (1): A lot of duplicates for this endpoint for some reason
# NOTE (2): The `pybraries` wrapper disabled this functionality

import time
from typing import Literal

import requests
import pandas as pd

platform: Literal["pypi", "conda"] = "pypi"

url = f"https://libraries.io/api/{platform}/pymatgen/dependent_repositories"

all_repos = []
page = 1

while True:
    print(f"Working on page {page}")
    time.sleep(1.1)  # keep under 60 requests/min

    resp = requests.get(url, params={"api_key": API_KEY, "per_page": 100, "page": page})

    if resp.status_code == 429:
        wait: int = int(resp.headers.get("Retry-After", 20))
        print(f"Rate limit hit, sleeping {wait}s…")
        time.sleep(wait)
        continue

    resp.raise_for_status()
    data = resp.json()
    if not data:
        break

    all_repos.extend(data)
    page += 1

df = pd.DataFrame(all_repos).drop_duplicates(subset=["full_name"])
df = df.sort_values("stargazers_count", ascending=False)

df.to_csv("dependents_repos_libraries.csv", index=False)
print(f"Saved {len(df)} repos to dependents_repos_libraries.csv")

Working on page 1
Working on page 2
Working on page 3
Working on page 4
Working on page 5
Working on page 6
Working on page 7
Working on page 8
Working on page 9
Working on page 10
Working on page 11
Working on page 12
Working on page 13
Working on page 14
Working on page 15
Working on page 16
Working on page 17
Working on page 18
Working on page 19
Working on page 20
Working on page 21
Working on page 22
Working on page 23
Working on page 24
Working on page 25
Working on page 26
Working on page 27
Working on page 28
Working on page 29
Working on page 30
Working on page 31
Working on page 32
Working on page 33
Working on page 34
Working on page 35
Working on page 36
Working on page 37
Working on page 38
Working on page 39
Working on page 40
Working on page 41
Working on page 42
Working on page 43
Saved 240 repos to pymatgen_dependents_repos.csv


In [ ]:
# Quickly preview the saved repos
import pandas as pd

df = pd.read_csv("dependents_repos_libraries.csv")

subset = df[["full_name", "homepage", "stargazers_count"]]

print(f"Total dependent repositories: {len(df)}\n")

# Show top rows
print(subset.head(10).to_string(index=False))

Total dependent repositories: 240

                full_name                                           homepage  stargazers_count
facebookresearch/fairchem                       https://fair-chem.github.io/            1684.0
      microsoft/mattergen https://www.nature.com/articles/s41586-025-08628-5            1466.0
                e3nn/e3nn                                                NaN            1143.0
     aiidateam/aiida-core                  https://aiida-core.readthedocs.io             492.0
hackingmaterials/matminer       https://hackingmaterials.github.io/matminer/             471.0
      microsoft/mattersim             https://microsoft.github.io/mattersim/             450.0
         SINGROUP/dscribe                https://singroup.github.io/dscribe/             409.0
 materialsvirtuallab/maml                                                NaN             404.0
       deepmodeling/dpgen      https://docs.deepmodeling.com/projects/dpgen/             353.0
     CederGroup